# Reinforcement Learning for Trade Execution with Market and Limit Orders
**ArXivist-generated reproduction notebook**

Paper: Cheridito & Weiss, arXiv:2507.06345 (https://arxiv.org/abs/2507.06345)
Generated: 2026-07-08

This notebook walks through the key components of the implementation (limit order
book, trader agents, logistic-normal policy, actor-critic trainer), runs a small-scale
training loop on the simulated market, and compares against the paper's reported
benchmark numbers.

In [ ]:
# Environment check
import sys, platform
print(f"Python: {sys.version}")
try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except ImportError as e:
    raise SystemExit(f"PyTorch not installed: {e}. Run the install cell below first.")
print(f"Using device: {device}")

In [ ]:
# Install the project in editable mode (run once)
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".."],
                         capture_output=True, text=True)
print(result.stdout[-2000:] if result.returncode == 0 else result.stderr[-2000:])

## Paper Overview

The paper formulates optimal **trade execution in a limit order book** as a dynamic
allocation problem: at each of $N$ discrete time steps, an algorithm selling $M$ lots
decides an allocation $a=(a_0,\dots,a_K)$ on the simplex $S^K$ across a market order
($a_0$), limit orders at $K-1$ price levels above the best bid ($a_1,\dots,a_{K-1}$),
and "do nothing" ($a_K$).

**Key innovation**: actions are modeled with a **logistic-normal distribution** on the
simplex (Section 4.2), enabling a closed-form actor-critic policy gradient (Eq. 9) that
is more effective than a Dirichlet-distribution baseline (Appendix B.2).

**Market simulation** (Section 5) includes noise traders (Poisson order flow), tactical
traders (imbalance-reactive order flow), and a strategic trader (large TWAP buyer/seller)
so that the algorithm's own trades generate realistic **direct and indirect market
impact** -- something historical-data-replay simulations cannot capture.

Code map:
- `src/rlte/env/order_book.py`, `traders.py`, `execution_env.py` &rarr; Sections 2, 3, 5
- `src/rlte/models/distributions.py`, `policy.py`, `value.py` &rarr; Section 4
- `src/rlte/training/trainer.py`, `losses.py` &rarr; Algorithm 1, Eq. 13-15
- `src/rlte/evaluation/benchmarks.py`, `metrics.py` &rarr; Section 3.4, Table 1

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))
import numpy as np
import torch
torch.manual_seed(0)
np.random.seed(0)

## Component 1: Logistic-Normal Transform (Section 4.2)

$$a_k = \frac{e^{x_k}}{1+\sum_{l=0}^{K-1}e^{x_l}},\ k=0,\dots,K-1; \qquad a_K = \frac{1}{1+\sum_{l=0}^{K-1}e^{x_l}}$$

Maps an unconstrained $x \in \mathbb{R}^K$ to an action on the simplex $S^K$.

In [ ]:
from rlte.models.distributions import LogisticNormalTransform

transform = LogisticNormalTransform()
x = torch.randn(2, 6)  # batch of 2, K=6
a = transform.forward(x)
x_recovered = transform.inverse(a)

print(f"x shape: {tuple(x.shape)}")
print(f"a shape: {tuple(a.shape)}  (K+1=7 dims, simplex)")
print(f"a sums to 1: {a.sum(dim=-1)}")
print(f"Round-trip error (x vs h^-1(h(x))): {(x - x_recovered).abs().max().item():.2e}")

## Component 2: Logistic-Normal Policy Network (Section 4.3, Appendix B.1)

2-hidden-layer tanh MLP (128 units) mapping normalized state features to the mean
$\mu_\theta(s)$ of the underlying normal distribution $N(\mu, \Sigma)$.

In [ ]:
from rlte.models.policy import LogisticNormalPolicy

state_dim = 32  # example; real dim depends on K, M, and D (see FeatureNormalizer)
K = 6
policy = LogisticNormalPolicy(state_dim=state_dim, K=K)
print(policy)
print(f"Param count: {sum(p.numel() for p in policy.parameters())}")

state = torch.randn(4, state_dim)  # batch of 4
sigma = 1.0  # initial variance, Eq. 12
a, x, mu = policy.sample(state, sigma)
print(f"Input state shape:  {tuple(state.shape)}")
print(f"Output action shape: {tuple(a.shape)}  (expected [4, 7])")
print(f"mu shape: {tuple(mu.shape)}  (expected [4, 6])")
det_a = policy.deterministic_action(state)
print(f"Deterministic action (eval mode): {det_a[0].detach().numpy().round(3)}")

## Component 3: Value Network (Critic, Section 4.4)

In [ ]:
from rlte.models.value import ValueNetwork

value_net = ValueNetwork(state_dim=state_dim)
v = value_net(state)
print(f"Value output shape: {tuple(v.shape)}  (expected [4, 1])")

## Component 4: Limit Order Book Simulator (Section 2, 5)

A discrete-price order book with FIFO queues per price level, populated by noise and
(optionally) tactical / strategic trader agents.

In [ ]:
from rlte.env.order_book import LimitOrderBook
from rlte.env.traders import NoiseTraders

book = LimitOrderBook(D=10, init_best_bid=1000.0, init_best_ask=1001.0)
noise = NoiseTraders(D=10)
rng = np.random.default_rng(0)

for _ in range(20):
    noise.step(book, dt=15.0, rng=rng)

bid_vols, ask_vols = book.get_volumes(5)
print(f"Best bid/ask after warm-up: {book.best_bid_ask()}")
print(f"Bid volumes (levels 1-5): {bid_vols}")
print(f"Ask volumes (levels 1-5): {ask_vols}")

## Mini Training Demo (synthetic-scale, runs on CPU)

We run a *tiny* version of Algorithm 1 (few envs, few iterations) purely to verify
the training loop is wired correctly end-to-end -- this is **not** meant to reproduce
paper-quality results (see the "Next Steps" section for full-scale training).

In [ ]:
from rlte.env.execution_env import MarketConfig
from rlte.training.trainer import ActorCriticTrainer, TrainConfig

market_cfg = MarketConfig(market_type="noise", D=10, T=60.0, dt=15.0, K=6, M=5)
train_cfg = TrainConfig(num_envs=2, traj_per_env=2, N=4, H=3, K=6, log_every=1)

try:
    trainer = ActorCriticTrainer(market_cfg, train_cfg, device="cpu")
    print(trainer.summary())
    result = trainer.train()
    print("Training history (avg return per iteration):", result["history"])
except Exception as e:
    print(f"Mini-training demo failed (this is a reference scaffold, not a fully "
          f"debugged production trainer -- see README 'Reproducibility Notes'): {e}")

## Paper's Reported Results (Table 1)

Reproduced here from the SIR for reference. To actually reproduce these numbers,
run the full-scale `train.py` + `evaluate.py` pipeline (computationally expensive;
paper reports ~1.2-2h per configuration on a 64-core/128-thread server, see README).

In [ ]:
paper_results = [
    {"market": "Noise", "lots": 20, "SL": 0.52, "TWAP": -0.06, "DR": 0.21, "LN": 0.61},
    {"market": "Noise", "lots": 60, "SL": -1.09, "TWAP": -1.40, "DR": -0.71, "LN": -0.72},
    {"market": "Noise & Tactical", "lots": 20, "SL": 0.10, "TWAP": 0.48, "DR": 0.73, "LN": 0.81},
    {"market": "Noise & Tactical", "lots": 60, "SL": -3.36, "TWAP": -0.96, "DR": -0.23, "LN": -0.25},
    {"market": "Noise & Tactical & Strategic", "lots": 20, "SL": -1.64, "TWAP": -0.36, "DR": 1.06, "LN": 1.13},
    {"market": "Noise & Tactical & Strategic", "lots": 60, "SL": -2.51, "TWAP": -1.45, "DR": 0.03, "LN": 0.23},
]
import pandas as pd
df = pd.DataFrame(paper_results)
print("Paper's reported E[normalized reward] (Table 1):")
print(df.to_string(index=False))
print("\nTo reproduce: python train.py --config configs/<market>_<lots>lots.yaml")
print("Then: python evaluate.py --policy LN --checkpoint runs/final_checkpoint.pt ...")
print("Then feed your results into ArXivist's Stage 6 Results Comparator.")

## What to do next

1. **Full training**: `python train.py --config configs/noise_20lots.yaml`
2. **Evaluation**: `python evaluate.py --policy LN --checkpoint runs/final_checkpoint.pt --market noise --lots 20`
3. **Single-episode visualization**: `python inference.py --checkpoint runs/final_checkpoint.pt`
4. **Compare results**: feed your `evaluate.py` output JSON back to ArXivist's Results Comparator (Stage 6)

**Top implementation assumptions to be aware of (see `sir.json` / README for full list):**
- Adam `beta1`/`beta2` assumed as PyTorch defaults (0.9/0.999) -- confidence 0.6, not stated in paper
- Exact simplex-to-lots rounding rule inferred as "sequential floor" -- confidence 0.6, SIR-flagged ambiguity
- Evaluation random seed assumed, not stated in paper -- confidence 0.4; report results as mean/std across seeds